# 🚀 07a — Deploy an agent from zero

*Before meeting this project's agents, build your own: an LLM call, then a hand-written
tool loop, then the same agent in LangGraph — so the real thing looks like your tutorial,
adapted.*

This chapter is deliberately different from the others: it is a **pure tutorial**. No
robberies, no contracts — just the honest answer to "if I had to deploy an agent myself,
starting from nothing, what would I actually type?" Everything here would work in any
project; chapter 07b then shows how *this* project adapts each piece.

**You need:** nothing. Every cell runs against a **scripted stub** — a fake LLM that
replays canned answers — which is exactly how this repo's own CI tests its agents
without a GPU. If you *do* have a live endpoint (`LLM_BASE_URL` pointing at Ollama,
vLLM, anything OpenAI-compatible), a few clearly-marked bonus cells will use it; without
one they skip politely.

**How to work through it:** run every cell in order; write your answer in each
**✏️ Your turn** scaffold before opening the fold-out solution. 🧭 Decision boxes and
the closing 📝 section work as in every chapter (00 explains the conventions).

## 0 · Where we are

Chapters 01–06 built the whole marketplace *except the shoppers*: a chain that can't
lie, offers that can't be forged, a vending machine that settles atomically, a bouncer
that authorizes deterministically, hands that configure the router. But every one of
those components only ever *reacts*. Nobody in the system has yet **wanted** anything —
nobody read "I need 50 Mbps for two hours," weighed a price, and chose.

That choosing is the agents' job, and it's where the LLM finally enters. So today's
question is the beginner one, asked honestly:

> **What actually *is* an agent, and how do I build and run one myself?**

## 1 · What *is* an agent? (strip the mystique)

Start with what an **LLM** is, mechanically: a function. Text goes in (a list of
messages), text comes out. That's the entire interface. It has no memory between calls,
no hands, no goals — it cannot check a balance, place an order, or remember what it said
a minute ago. Calling it twice with the same input is just... calling a function twice.

To make the type concrete before any real model is involved:

In [ ]:
def pretend_llm(messages):
    """An 'LLM' with one hardcoded thought. Wrong, but the right SHAPE:
    list-of-messages in, string out. Every real model has this interface."""
    return "I would check the budget first."

reply = pretend_llm([
    {"role": "system", "content": "You are a procurement assistant."},
    {"role": "user",   "content": "Buy 50 Mbps of bandwidth if we can afford it."},
])
print(reply)

An **agent** is not a smarter model — it is a *loop wrapped around that function* that
lets its words cause effects:

1. show the model the current state of the world (as text),
2. let it **choose an action** (also text — "call this tool with these arguments"),
3. **execute** the action with ordinary code,
4. append the result to the conversation, and go to 1 — until the model says it's done.

Everything sold under the word "agent" — chatbots that browse, coding assistants,
Ada and Bell — is this loop with better ingredients. We'll now build the ingredients
one at a time: the real call (§2), the loop (§3), the loop drawn as a graph (§4), and
the discipline that makes the words safe to act on (§5).

## 2 · The bare LLM call — one wire format to rule them all

Almost every LLM server on earth — OpenAI's, a local Ollama, a vLLM box in your lab, the
Modal deployment this project uses — speaks the same HTTP dialect, called
**OpenAI-compatible** after the company that popularized it. The request and response
are plain JSON, and it's worth seeing them as plain data once, with no library in the
way:

In [ ]:
import json

request = {
    "model": "qwen3:4b",                       # which model the server should run
    "temperature": 0.0,                        # 0 = deterministic-ish; decisions want this
    "messages": [                              # the whole conversation so far
        {"role": "system", "content": "You are a procurement assistant."},
        {"role": "user",   "content": "Buy 50 Mbps if we can afford it."},
    ],
}

response = {                                   # what any compatible server sends back
    "choices": [
        {"message": {"role": "assistant", "content": "I would check the budget first."}}
    ],
    "usage": {"prompt_tokens": 31, "completion_tokens": 8},
}

print(json.dumps(request, indent=2)[:400])
print("assistant said:", response["choices"][0]["message"]["content"])

That's the entire protocol: POST that request to `<base_url>/chat/completions`, read
`choices[0].message.content`. The `openai` Python package is a convenience wrapper
around exactly this — and pointing it at a *different* `base_url` is how one client
talks to any backend.

> **🧭 Decision (principled) — agents speak one client shape, chosen by environment**
>
> **Chosen:** every agent reaches its model through the generic OpenAI-compatible
> client, configured by three environment variables (`LLM_BASE_URL`, `LLM_MODEL`,
> `LLM_API_KEY`) — ADR-001. No other LLM SDK is ever imported by agent code.
> **Alternatives:** (a) a vendor SDK per backend (an `ollama` import here, a cloud SDK
> there); (b) a framework's model-wrapper layer for everything.
> **Why:** the backend genuinely changed during this project's life — local Ollama on
> the lab PC, vLLM on Modal for the evaluation, a scripted stub in CI — and with one
> client shape those swaps are an *environment edit, zero code*. Tests run without a
> GPU for the same reason. (b) buys the same portability but adds a dependency layer
> the two judgment slots are too small to need.
> **In the paper:** §5.1 — "an OpenAI-compatible client", one clause; the decision's
> real payoff is that the deterministic baseline and the stub-run tests exist at all.

Is a live endpoint around right now? Let's probe — and set up this chapter's guard
(same pattern as 03's `CHAIN_OK`):

In [ ]:
from agents.llm import LLMConfig, llm_up

CONFIG = LLMConfig.from_env()      # reads the three env vars, with local-Ollama defaults
LLM_OK = llm_up(CONFIG)            # a bounded 1-token probe, not a mere ping
SKIP_LLM = "skipped: no live endpoint at LLM_BASE_URL — the stub cells cover the logic"

print("LLM_BASE_URL →", CONFIG.base_url)
print("LLM_MODEL    →", CONFIG.model)
print("live         →", "✓" if LLM_OK else "✗ (fine — everything below runs on the stub)")

In [ ]:
# BONUS (live endpoint only): the same wire format, for real.
if not LLM_OK:
    print(SKIP_LLM)
else:
    from openai import OpenAI
    client = OpenAI(base_url=CONFIG.base_url, api_key=CONFIG.api_key)
    r = client.chat.completions.create(
        model=CONFIG.model, temperature=0.0,
        messages=[{"role": "system", "content": "Answer in five words or fewer."},
                  {"role": "user", "content": "What does an agent loop do?"}],
    )
    print("assistant said:", r.choices[0].message.content)

And here is the tutorial's workhorse: a **scripted stub**. It has the same shape as a
model — messages in, string out — but replays answers *we* wrote, one per call. Two
honest reasons to love it: your notebook runs identically on any machine, and you can
script the *failure* cases a real model only produces when it feels like it. This is
exactly how the repo's CI pins its agent logic without a model server.

In [ ]:
def scripted_llm(replies):
    """Make a fake LLM that replays `replies` in order. Same interface as pretend_llm."""
    replies = list(replies)
    calls = {"n": 0}
    def fake(messages):
        reply = replies[calls["n"]]
        calls["n"] += 1
        return reply
    return fake

demo = scripted_llm(["first canned answer", "second canned answer"])
print(demo([{"role": "user", "content": "anything"}]))
print(demo([{"role": "user", "content": "anything"}]))

## 3 · A tool-calling loop, written by hand

Now the loop. To act, the model needs **tools**: functions we let it call. We describe
them in the system prompt and ask the model to answer *in a format our code can parse* —
either a tool call or a final answer:

```
TOOL: {"name": "check_budget", "args": {}}
— or —
FINAL: <its answer to the human>
```

(One honesty note: modern APIs have a native `tools=` field that returns structured
tool calls, so real code rarely parses prefixes out of raw text. We hand-roll the
visible version because it's the same idea with nothing hidden — and §5's *validate
everything* discipline applies identically to both.)

Our toy agent is a tiny procurement clerk with two tools:

In [ ]:
BUDGET_TOK = 12

def check_budget():
    return {"budget_tok": BUDGET_TOK}

def place_order(item, price_tok):
    if price_tok > BUDGET_TOK:
        return {"ok": False, "error": "over budget"}
    return {"ok": True, "order": f"{item} @ {price_tok} TOK", "confirmation": "ORD-7"}

TOOLS = {"check_budget": check_budget, "place_order": place_order}

SYSTEM = """You are a procurement agent. You may use tools by replying EXACTLY:
TOOL: {"name": "<tool>", "args": {...}}
Available tools:
  check_budget()                      -> the budget in TOK
  place_order(item, price_tok)        -> places the order
When finished, reply:
FINAL: <one-sentence summary for your principal>"""
print(SYSTEM)

In [ ]:
import json

def run_agent(llm, task, max_steps=6):
    """The agent loop, version 1: call, parse, execute, feed back, repeat."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        reply = llm(messages)
        messages.append({"role": "assistant", "content": reply})
        print(f"step {step} | model: {reply}")

        if reply.startswith("FINAL:"):
            return reply.removeprefix("FINAL:").strip()

        call = json.loads(reply.removeprefix("TOOL:").strip())   # <- trusts the model!
        result = TOOLS[call["name"]](**call["args"])
        messages.append({"role": "user", "content": f"TOOL RESULT: {json.dumps(result)}"})
        print(f"       | tool {call['name']} -> {result}")
    return "(gave up: too many steps)"

Drive it with a scripted trace — the happy path a competent model would produce for
"buy 50 Mbps at 10 TOK if we can afford it":

In [ ]:
happy_trace = [
    'TOOL: {"name": "check_budget", "args": {}}',
    'TOOL: {"name": "place_order", "args": {"item": "50 Mbps, 14:00-16:00", "price_tok": 10}}',
    'FINAL: Ordered 50 Mbps for 10 TOK (confirmation ORD-7); 2 TOK of budget remain.',
]

answer = run_agent(scripted_llm(happy_trace), "Buy 50 Mbps at 10 TOK if we can afford it.")
print("\nagent returned:", answer)

Perceive → choose → execute → feed back: that's a working agent in ~20 lines, and every
framework demo you've ever seen is this loop in nicer clothes.

**Now the pains** — earned the usual way, by watching them happen.

**Pain (a): the model's words are not code.** Version 1 does
`json.loads(...)` straight off the model's reply. A model — especially a small one —
*will* eventually emit something that isn't quite JSON. Watch:

In [ ]:
clumsy_trace = [
    'TOOL: {"name": "check_budget", "args": {}}',
    "TOOL: {'name': 'place_order', 'args': {'item': '50 Mbps', 'price_tok': 10}}",  # single quotes!
]

try:
    run_agent(scripted_llm(clumsy_trace), "Buy 50 Mbps at 10 TOK if we can afford it.")
except Exception as e:
    print(f"\n💥 the whole agent crashed: {type(e).__name__}: {e}")

One almost-right reply and the agent is a stack trace. The fix is the single most
important discipline in this whole chapter — **validate, retry once with the error
shown, then fail SAFE**:

In [ ]:
def run_agent_v2(llm, task, max_steps=8):
    """v2: never trust the words. Parse defensively; on garbage, show the model its
    error and retry ONCE; if it still can't produce a valid call, abort — placing no
    order is always safer than placing a mangled one."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    retried = False
    for step in range(1, max_steps + 1):
        reply = llm(messages)
        messages.append({"role": "assistant", "content": reply})
        print(f"step {step} | model: {reply}")

        if reply.startswith("FINAL:"):
            return reply.removeprefix("FINAL:").strip()
        try:
            call = json.loads(reply.removeprefix("TOOL:").strip())
            tool, args = TOOLS[call["name"]], call["args"]
        except Exception as e:
            if retried:
                return "(aborted: could not get a valid tool call — no order was placed)"
            retried = True
            messages.append({"role": "user",
                             "content": f"Your reply was not valid ({e}). Reply again, exactly in the required format."})
            print(f"       | ⚠ invalid ({type(e).__name__}) — retrying once")
            continue
        retried = False
        result = tool(**args)
        messages.append({"role": "user", "content": f"TOOL RESULT: {json.dumps(result)}"})
        print(f"       | tool {call['name']} -> {result}")
    return "(gave up: too many steps)"

# The clumsy model, given one nudge, gets it right:
recovering_trace = [
    'TOOL: {"name": "check_budget", "args": {}}',
    "TOOL: {'name': 'place_order', 'args': {'item': '50 Mbps', 'price_tok': 10}}",   # bad
    'TOOL: {"name": "place_order", "args": {"item": "50 Mbps", "price_tok": 10}}',   # fixed
    'FINAL: Ordered 50 Mbps for 10 TOK.',
]
print(run_agent_v2(scripted_llm(recovering_trace), "Buy 50 Mbps at 10 TOK."), "\n")

# And a model that NEVER gets it right is contained, not obeyed:
hopeless_trace = ["TOOL: gibberish", "TOOL: more gibberish"]
print(run_agent_v2(scripted_llm(hopeless_trace), "Buy 50 Mbps at 10 TOK."))

Hold on to that last line: **persistent garbage degrades into a safe no-op, never a
crash and never a guess.** In 07b you'll find this exact pattern, verbatim, at both of
the project's judgment slots (a decision the paper states in §5.1: *"persistent
malformation fails safe"*).

Two more pains, visible in the code you just wrote rather than run:

- **Pain (b): state threading.** Our loop hand-carries everything in one `messages`
  list plus stray variables (`retried`, budgets, step counts). Add three more tools, a
  branch ("if declined, try the other provider"), and a resume-after-crash requirement,
  and this function becomes spaghetti — every new concern threads through the same body.
- **Pain (c): opacity.** While the loop runs, there is no way to see *where* it is, stop
  it at a checkpoint, or draw its possible paths for a colleague. The control flow
  exists only as Python's instruction pointer.

These two pains — not intelligence — are what a graph framework organizes.

**✏️ Your turn 1 — a third tool**

Give the clerk a `cancel_order(confirmation)` tool: add it to `TOOLS` and to the
`SYSTEM` prompt, then script a trace where the model orders, thinks better of it
(pretend the principal changed their mind in the task text), cancels, and reports.
Run it through `run_agent_v2`.

In [ ]:
def cancel_order(confirmation):
    return {"ok": True, "cancelled": confirmation}

# TOOLS["cancel_order"] = ...
# SYSTEM_V2 = SYSTEM + ...        (or edit SYSTEM before re-running the loop cell)
# change_of_heart_trace = [ ... ]
# print(run_agent_v2(scripted_llm(change_of_heart_trace), "Order 50 Mbps, then cancel it."))

<details><summary>✅ Solution 1 — peek only after trying</summary>

```python
TOOLS["cancel_order"] = cancel_order
SYSTEM += "\n  cancel_order(confirmation)          -> cancels a placed order"

change_of_heart_trace = [
    'TOOL: {"name": "place_order", "args": {"item": "50 Mbps", "price_tok": 10}}',
    'TOOL: {"name": "cancel_order", "args": {"confirmation": "ORD-7"}}',
    'FINAL: Ordered, then cancelled per your instruction; no funds committed.',
]
print(run_agent_v2(scripted_llm(change_of_heart_trace), "Order 50 Mbps, then cancel it."))
```

Note what adding one tool touched: the registry, the prompt, and (in your head) every
trace. Tools are cheap; *keeping the model well-briefed about them* is the real
maintenance cost.

</details>

## 4 · The same agent, drawn: LangGraph in ~35 lines

A graph framework asks you to make the loop's anatomy *explicit*: a **state** object
(all those stray variables, in one declared place), **nodes** (functions state flows
through), and **edges** (which node runs next — including conditional branches). Then
it runs the graph for you. The one below is our §3 clerk, reorganized — same stub, same
tools, same behavior:

In [ ]:
from dataclasses import dataclass, field
from langgraph.graph import END, StateGraph

@dataclass
class ClerkState:                       # pain (b), solved: ALL state, declared once
    task: str
    messages: list = field(default_factory=list)
    done: bool = False
    answer: str = ""

def build_clerk(llm):
    def call_model(state: ClerkState) -> ClerkState:
        if not state.messages:
            state.messages = [{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": state.task}]
        reply = llm(state.messages)
        state.messages.append({"role": "assistant", "content": reply})
        if reply.startswith("FINAL:"):
            state.done, state.answer = True, reply.removeprefix("FINAL:").strip()
        return state

    def run_tool(state: ClerkState) -> ClerkState:
        call = json.loads(state.messages[-1]["content"].removeprefix("TOOL:").strip())
        result = TOOLS[call["name"]](**call["args"])
        state.messages.append({"role": "user", "content": f"TOOL RESULT: {json.dumps(result)}"})
        return state

    graph = StateGraph(ClerkState)
    graph.add_node("model", call_model)
    graph.add_node("tool", run_tool)
    graph.set_entry_point("model")
    graph.add_conditional_edges("model", lambda s: "end" if s.done else "tool",
                                {"end": END, "tool": "tool"})
    graph.add_edge("tool", "model")     # after a tool, always ask the model again
    return graph.compile()

clerk = build_clerk(scripted_llm(happy_trace))
final = clerk.invoke(ClerkState(task="Buy 50 Mbps at 10 TOK if we can afford it."))
print("answer :", final["answer"])
print("turns  :", len(final["messages"]), "messages in the transcript")

Nothing got smarter — the model is the same stub. What changed is *shape*:

| §3 hand loop | LangGraph version |
|---|---|
| the `while`/`for` around everything | the graph itself (`tool → model` edge closes the loop) |
| `messages` list + stray variables | one declared `ClerkState` |
| the `if reply.startswith("FINAL:")` | the conditional edge out of `model` |
| the tool-dispatch block | the `tool` node |
| "where is it now?" — unanswerable | a named node, inspectable/streamable per step |

> **🧭 Decision (pragmatic) — LangGraph, and it could have been otherwise**
>
> The project's two agents are LangGraph workflows. Honestly: at this size, the §3 hand
> loop would have worked — the consumer's whole journey is six fixed steps with one
> branch. LangGraph was chosen because the workflow *is* naturally a small explicit
> state machine (nodes = lifecycle steps, one conditional edge at the decision), and
> because its ecosystem and documentation made it the lowest-friction way to get
> exactly that. Another framework, or none, would also have served; nothing in the
> results depends on the choice. What the paper *does* lean on is **where judgment
> sits** — which node may call the LLM — and that boundary is framework-independent.
> **In the paper:** §5.1 cites LangGraph as scaffolding, one word, no argument built
> on it.

**✏️ Your turn 2 — a give-up edge**

Our graph loops `tool → model` forever if the model never says `FINAL:` (the compiled
graph has a recursion limit that would eventually stop it with an error — try scripting
an endless trace if you're curious). Add a proper guard instead: a `steps` counter in
`ClerkState`, incremented in `call_model`, and a third branch in the conditional edge —
after 4 model calls, route to a `give_up` node that sets a safe answer. Script a stub
that dithers (keeps calling `check_budget`) and watch the graph bail out gracefully.

In [ ]:
# @dataclass
# class ClerkState2: ...            # add: steps: int = 0
# def build_clerk2(llm): ...        # add give_up node + three-way conditional edge
# dithering = scripted_llm(['TOOL: {"name": "check_budget", "args": {}}'] * 10)
# print(build_clerk2(dithering).invoke(ClerkState2(task="Buy 50 Mbps."))["answer"])

<details><summary>✅ Solution 2 — peek only after trying</summary>

```python
@dataclass
class ClerkState2(ClerkState):
    steps: int = 0

def build_clerk2(llm):
    def call_model(state):
        state.steps += 1
        if not state.messages:
            state.messages = [{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": state.task}]
        reply = llm(state.messages)
        state.messages.append({"role": "assistant", "content": reply})
        if reply.startswith("FINAL:"):
            state.done, state.answer = True, reply.removeprefix("FINAL:").strip()
        return state

    def run_tool(state):
        call = json.loads(state.messages[-1]["content"].removeprefix("TOOL:").strip())
        result = TOOLS[call["name"]](**call["args"])
        state.messages.append({"role": "user", "content": f"TOOL RESULT: {json.dumps(result)}"})
        return state

    def give_up(state):
        state.answer = "(gave up after 4 model calls — no order was placed)"
        return state

    graph = StateGraph(ClerkState2)
    graph.add_node("model", call_model); graph.add_node("tool", run_tool)
    graph.add_node("give_up", give_up)
    graph.set_entry_point("model")
    graph.add_conditional_edges(
        "model",
        lambda s: "end" if s.done else ("give_up" if s.steps >= 4 else "tool"),
        {"end": END, "tool": "tool", "give_up": "give_up"})
    graph.add_edge("tool", "model"); graph.add_edge("give_up", END)
    return graph.compile()

dithering = scripted_llm(['TOOL: {"name": "check_budget", "args": {}}'] * 10)
print(build_clerk2(dithering).invoke(ClerkState2(task="Buy 50 Mbps."))["answer"])
```

The guard is an *edge*, not an `if` buried in a loop body — visible in the graph's
drawing, testable on its own. That legibility is most of what the framework buys.

</details>

## 5 · Structured judgment — making words safe to act on

Here is the move that turns this tutorial into *this project's* agents. Look at what
Ada and Bell actually need their LLM for — and how little it is:

- Bell, holding a request and his price list: **quote or decline?**
- Ada, holding a signed offer, her need, and a budget: **accept or reject?**

Not open-ended conversation — a **bounded, structured verdict**: a yes/no plus a reason,
shaped exactly so. The pattern for that is §3's discipline, upgraded from "parseable"
to "schema-valid":

1. tell the model the JSON schema its answer must match,
2. **validate** the reply against that schema (pydantic — chapter 01's border guard,
   now guarding the model),
3. on failure, retry with a fresh ask,
4. out of retries → **fail safe**: reject/decline, never crash, never guess.

The repo implements this once, in `agents/src/agents/llm.py` (`LLMClient.structured`),
and both judgment slots go through it. Let's run the *real* class — on our scripts.
The trick (borrowed from the repo's own tests): the client's inner OpenAI object can be
swapped for a scripted one, because it only ever calls `chat.completions.create(...)`:

In [ ]:
from pydantic import BaseModel

from agents.llm import LLMClient, LLMConfig, StructuredError

class ScriptedChat:
    """Stands in for the OpenAI client's chat.completions — replays canned replies.
    This is exactly how agents/tests/test_llm_retry.py pins the retry loop in CI."""
    def __init__(self, replies):
        self._replies, self.calls = list(replies), 0
    def create(self, **_kwargs):
        reply = self._replies[self.calls]; self.calls += 1
        return type("R", (), {"choices": [type("C", (), {
            "message": type("M", (), {"content": reply})})]})

def real_client_with(replies):
    client = LLMClient(LLMConfig(base_url="stub", model="stub", api_key="stub"))
    client._client = type("O", (), {"chat": type("Ch", (), {
        "completions": ScriptedChat(replies)})})()
    return client

class Verdict(BaseModel):
    accept: bool
    reason: str

In [ ]:
# A well-behaved model: schema-valid on the first try.
client = real_client_with(['{"accept": true, "reason": "meets the need, within budget"}'])
v = client.structured("You are a procurement agent.", "NEED ... OFFER ...", Verdict)
print("verdict  :", v, f"   (attempts: {client.last_attempts})")

# A wandering small model: prose, then a missing field, then finally valid.
client = real_client_with([
    "Sure! I think you should probably accept this offer.",     # prose, not JSON
    '{"accept": true}',                                         # JSON, missing `reason`
    '{"accept": true, "reason": "capacity and window match"}',  # valid at last
])
v = client.structured("You are a procurement agent.", "NEED ... OFFER ...", Verdict)
print("verdict  :", v, f"   (attempts: {client.last_attempts} — it retried in code)")

# A hopeless model: three strikes -> a clean, catchable exception. Never a guess.
client = real_client_with(["nope", "still nope", "nope again"])
try:
    client.structured("You are a procurement agent.", "NEED ... OFFER ...", Verdict)
except StructuredError as e:
    print("hopeless :", e, " — every attempt kept for the post-mortem:", len(e.attempts))

Two lovely production details worth noticing in the real class, because you now have
the context to appreciate them:

- **It scrubs the reply before parsing.** Small models wrap answers in ` ```json `
  fences, or think out loud in `<think>…</think>` blocks first. `_extract_json` peels
  all of that and takes the first balanced `{…}` — the schema check then judges what's
  left. Robustness lives in *our* code, not in hoping the model behaves:

In [ ]:
from agents.llm import _extract_json

for raw in ['```json\n{"accept": true, "reason": "ok"}\n```',
            '<think>hmm, price is fine, window matches...</think>\n{"accept": true, "reason": "ok"}',
            'here you go: {"accept": true, "reason": "ok"} hope that helps!']:
    print(repr(raw[:38]), "→", _extract_json(raw))

- **And the caller adds the last belt: a domain-correct safe default.** Ada's real
  decision slot (`agents/src/agents/decision.py`) wraps `structured()` like this —
  quoted, because it's four lines:

  ```python
  try:
      return client.structured(_SYSTEM, user, DecisionOutput)
  except StructuredError:
      return DecisionOutput(accept=False, reason="could not obtain a valid decision; declining")
  ```

  Run it — the real project function, on a hopeless model, deciding about the real
  canonical offer:

In [ ]:
from a2a_interfaces import fixtures as fx
from agents.decision import decide

hopeless = real_client_with(["garbage", "more garbage", "even more garbage"])
decision = decide(hopeless, fx.BANDWIDTH_NEED, fx.CANONICAL_SIGNED_OFFER, budget_tok=12)
print(decision)
print("\nA procurement agent that cannot READ an offer must never ACCEPT one.")

> **🧭 Decision (principled) — a small model is enough, because the slots are bounded**
>
> **Chosen:** a small open model (Qwen3-4B, served by vLLM; Ollama on the lab PC)
> behind the two judgment slots.
> **Alternatives:** (a) a large API model; (b) fine-tuning a model for the task.
> **Why:** the slots are *bounded structured tasks* — read a need, an offer, a budget;
> emit one schema-valid verdict — not open-ended reasoning; and the fail-safe wrapper
> means a wrong-shaped answer degrades to a decline, never to damage. Under that
> contract a 4B model is sufficient (ch. 09 measures it: schema-valid quotes 10/10,
> correct decisions on 12/12 curated cases), runs on lab hardware with no external
> dependency, and keeps the whole negotiation under a cent of tokens. (a) buys latency,
> cost, and an internet dependency for headroom the slots don't use; (b) solves a
> problem we never observed.
> **In the paper:** §5.1, and RQ3's token-cost line in §7.3. The honest edge (§8.4):
> those accuracy numbers are curated and single-sample — a robustness study, not this
> demonstration.

**✏️ Your turn 3 — predict the retry count**

Script the real `LLMClient` so that it succeeds on the **third and final** attempt
(two invalid replies, then a valid `Verdict`). *Before running*: what will
`client.last_attempts` be, and what happens if your third reply is invalid too?

In [ ]:
# client = real_client_with([ ... , ... , ... ])
# v = client.structured("sys", "usr", Verdict)
# print(v, client.last_attempts)

<details><summary>✅ Solution 3 — peek only after trying</summary>

```python
client = real_client_with([
    "let me think about this one...",          # invalid: prose
    '{"accept": "definitely"}',                # invalid: wrong type, missing field
    '{"accept": false, "reason": "price exceeds budget"}',
])
v = client.structured("sys", "usr", Verdict)
print(v, client.last_attempts)                  # last_attempts == 3
```

`last_attempts` is 3 — the retry budget (`max_retries=3`) exactly spent. Make the third
reply invalid too and `structured()` raises `StructuredError` carrying all three
attempts; a caller like `decide()` then returns its safe decline. There is no fourth
chance by design: retrying forever would let a broken model stall the whole purchase.

</details>

**✏️ Your turn 4 — the dangerous default (a thought you run)**

In `decide()`'s four quoted lines, flip the fail-safe to
`DecisionOutput(accept=True, reason="model was down, assuming fine")` — in a copy, in
your head, or in a scratch cell. Then write one sentence: what can now happen that
could not happen before?

In [ ]:
# my_sentence = "..."

<details><summary>✅ Solution 4 — peek only after trying</summary>

One possible sentence: *a broken or unreachable model now causes the agent to
**spend money on an offer nobody evaluated** — the failure mode changed from "no deal
happens" (annoying, free) to "an unexamined deal happens" (silent, costly, and
repeatable at machine speed).* Fail-safe direction is a one-word choice with an
asymmetric blast radius; every judgment slot in this project fails toward *not acting*.

</details>

## 6 · Where the LLM must never be

You now hold both halves of the boundary that chapter 05 drew from the other side:

- **Judgment** — should Bell quote 10 TOK? should Ada take the deal? — is fuzzy,
  contextual, and *safe to get wrong* (a bad quote loses a sale; a bad accept wastes
  10 TOK). That's where the LLM lives: **exactly two slots**, both structured, both
  fail-safe.
- **Authorization** — does ticket #7 admit its holder *right now*? — must be
  reproducible, auditable, and immune to a cleverly-worded request. That's the
  bouncer's six deterministic checks, and no words reach it: by the time the predicate
  runs, everything is signatures, ownership, and chain time.

*Judgment decides whether to buy; arithmetic decides whether you get in.* Chapter 07b
opens the real Ada and Bell and points at the two slots — you'll recognize every part:
the graph is §4, the slots are §5, the stub is how we'll drive them.

## 7 · What you can now say

- **What an agent is:** an LLM (a text function) wrapped in a loop that shows it state,
  parses its chosen action, executes with ordinary code, and feeds results back.
- **What the wire format is:** OpenAI-compatible JSON over `chat/completions` — one
  client shape for cloud, local vLLM/Ollama, or a scripted stub.
- **Why validation is the load-bearing wall:** you crashed v1 with one almost-JSON
  reply; v2's validate-retry-fail-safe contained a hopeless model. The real
  `LLMClient.structured` is that pattern with a schema.
- **What a graph framework buys:** not intelligence — declared state, named nodes,
  visible edges. Your §3 loop and your §4 graph did identical work.
- **Why fail-safe direction matters:** exercise 4's one-word flip turned "no deal" into
  "unexamined deal."

**Loose threads, on purpose:** the real Ada/Bell graphs, the two slots in situ, the
capacity ledger, and how agents *talk to each other* (A2A) and to their key custodians
(MCP) — all chapter **07b**.

## 8 · 📝 For the paper

Where this chapter lands: the background one-liners of **§2.1** (LLM agents, tool
loops, orchestration frameworks, MCP) and the implementation facts of **§5.1** (agent
workflow, small model, fail-safe slots). Draft sentences, each with the evidence you ran:

| you can write… | because you ran… |
|---|---|
| *Both agents are LangGraph workflows reaching a small model through an OpenAI-compatible client configured entirely by environment, so the same agent code runs against vLLM, a local Ollama, or a scripted stub in CI.* | §2's wire-format cells + the stub driving the identical loop in §3–§4 |
| *Each LLM slot validates the model's structured output against a schema and retries in code; persistent malformation fails safe to a decline/reject, never a crash and never an unexamined acceptance.* | §5: the real `LLMClient` retrying (attempts=3), `StructuredError` on a hopeless model, and the real `decide()` declining on garbage |
| *A small model suffices because the two judgment slots are bounded structured tasks, not open-ended reasoning.* | the entire chapter: every agent behavior you built was driven by canned text — the *loop and its guards*, not model brilliance, carried the logic |

**Reviewer objections you can now answer:** *"Why such a small model?"* — the slots'
contract (bounded task + schema guard + safe default) is doing the safety work; model
size buys margin, not correctness (and ch. 09 reports the measured accuracy, curated
cases and all). *"What if the LLM emits garbage?"* — you scripted garbage three ways;
the system's answer was retry-then-safe-decline every time. *"Is the contribution the
agent framework?"* — no, and say so plainly: the contribution is **where the LLM is
and isn't** (two buying-side slots; zero authorization-side), which any framework — or
§3's bare loop — could host.

**Honesty inventory for §8.4:** every trace in this chapter was scripted (that's the
point — it pins logic, not model quality); the live accuracy and token numbers are
chapter 09's, and they are curated and single-sample by the paper's own admission.

*Next: [07b — This project's agents](07b_this_projects_agents.ipynb)*